# Qwen3-VL-4B 4-Task Structured Decoding Pipeline

Reproduce the Qwen2-VL `7-refactor-order` 4-task setup with Qwen3-VL-4B-Instruct.

Goals:

- Use the same Id split as the best Qwen2-VL reference run.
- Use only four tasks: `order / pairwise / first / last`.
- Keep the same task ratio and loss weight.
- Keep the existing `pairwise + first + last` structured decoder.
- Exclude endpoint, adjacent, and multiturn tasks from this first experiment.
- Build pairwise candidates in both directions at pool level and assert a 50:50 train label distribution.


In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_qwen3vl_4b_4task_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.57.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")


In [ ]:
# 2) Setup + data unzip
from google.colab import drive
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import itertools
import json
import math
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
import random
import re
import shutil
import zipfile
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
try:
    from transformers import Qwen3VLForConditionalGeneration
except ImportError:
    Qwen3VLForConditionalGeneration = None
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None
try:
    from transformers import AutoModelForVision2Seq
except ImportError:
    AutoModelForVision2Seq = None
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")
warnings.filterwarnings("ignore", message=".*The following generation flags are not valid.*")
transformers_logging.set_verbosity_error()

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

MODEL_REPO_ID = "Qwen/Qwen3-VL-4B-Instruct"
USE_MODELSCOPE_BASE_MODEL = True
DRIVE_MODEL_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/model_cache/Qwen3-VL-4B-Instruct"


def ensure_base_model_path():
    if os.path.exists(os.path.join(DRIVE_MODEL_DIR, "config.json")):
        print("Using cached base model:", DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR

    if not USE_MODELSCOPE_BASE_MODEL:
        print("Using Hugging Face repo id:", MODEL_REPO_ID)
        return MODEL_REPO_ID

    print("Base model cache not found. Downloading via ModelScope:")
    print(DRIVE_MODEL_DIR)
    from modelscope import snapshot_download as modelscope_snapshot_download

    model_dir = modelscope_snapshot_download(
        MODEL_REPO_ID,
        cache_dir="/content/modelscope_cache",
    )
    os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)
    if not os.path.exists(DRIVE_MODEL_DIR):
        shutil.copytree(model_dir, DRIVE_MODEL_DIR)
    print("Base model cached at:", DRIVE_MODEL_DIR)
    return DRIVE_MODEL_DIR


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = os.path.isdir(MODEL_ID)
OUTPUT_ROOT = "/content/drive/MyDrive/SNU_AI_Challenge/qwen3vl_4b_4task_structured_v1"
SPLIT_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/id_splits/qwen2vl_lgt_order_refine_20260714_003635"
QWEN2_BEST_ADAPTER_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/best_adapter"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = os.path.join(OUTPUT_ROOT, "runs", RUN_ID)
OUTPUT_DIR = os.path.join(RUN_ROOT, "qwen3vl_4b_4task")
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
BEST_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "best_adapter")
SUBMIT_PATH = os.path.join(OUTPUT_DIR, "submission_qwen3vl_4b_4task.csv")
RUN_SPLIT_DIR = os.path.join(RUN_ROOT, "splits")

SEED = 42
VALID_RATIO = 0.10
TRAIN_ROWS = None
VALID_ROWS = None

TASK_RATIOS = {
    "order": 0.40,
    "pairwise": 0.20,
    "first": 0.20,
    "last": 0.20,
}
TASK_LOSS_WEIGHTS = {
    "order": 1.0,
    "pairwise": 1.0,
    "first": 1.0,
    "last": 1.0,
}

LEARNING_RATE = 1e-5
MAX_TRAIN_STEPS = -1
SAVE_STEPS = 100
LOGGING_STEPS = 20
QUICK_EVAL_ROWS = 50
TUNING_EVAL_ROWS = 150
HOLDOUT_EVAL_ROWS = 150
FULL_EVAL_ROWS = TUNING_EVAL_ROWS + HOLDOUT_EVAL_ROWS
TOP_K_FULL_EVAL = 3

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))
ALPHAS = [0.5, 1.0, 1.5, 2.0]
BETAS = [0.5, 1.0, 1.5, 2.0]
GAMMAS = [0.5, 1.0, 1.5, 2.0]
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

for path in [OUTPUT_ROOT, RUN_ROOT, OUTPUT_DIR, EVAL_DIR, SPLIT_DIR, RUN_SPLIT_DIR]:
    os.makedirs(path, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("model:", MODEL_ID)
print("shared split dir:", SPLIT_DIR)
print("qwen2 best adapter:", QWEN2_BEST_ADAPTER_DIR)
print("output:", OUTPUT_DIR)


In [ ]:
# 3) Data split + multitask record generation
def save_json(data, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def format_order(order):
    return "[" + ", ".join(str(int(value)) for value in order) + "]"


def parse_order_prediction(text):
    match = re.fullmatch(r"\s*\[\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*\]\s*", str(text))
    if not match:
        return None
    values = [int(value) for value in match.groups()]
    return values if sorted(values) == [1, 2, 3, 4] else None


def row_image_paths(row, image_root):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def pairwise_target(answer, first_index, second_index):
    return "1" if int(answer[first_index]) < int(answer[second_index]) else "2"


def base_record(row_index, row):
    answer = [int(value) for value in row["Answer_list"]]
    order = order_to_sequence(answer)
    return {
        "row_index": int(row_index),
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order,
        "image_paths": row_image_paths(row, TRAIN_IMAGE_DIR),
    }


def make_pairwise_item(base, candidate_a, candidate_b):
    item = copy.deepcopy(base)
    item.update({
        "task_type": "pairwise",
        "first_index": candidate_a,
        "second_index": candidate_b,
        "candidate_order": [candidate_a + 1, candidate_b + 1],
        "image_paths": [base["image_paths"][candidate_a], base["image_paths"][candidate_b]],
        "target": pairwise_target(base["answer"], candidate_a, candidate_b),
    })
    return item


def build_record_pools(dataframe):
    pools = {task: [] for task in TASK_RATIOS}
    for row_index, row in dataframe.iterrows():
        base = base_record(row_index, row)

        item = copy.deepcopy(base)
        item.update({"task_type": "order", "target": format_order(base["order"])})
        pools["order"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "first", "target": str(base["order"][0])})
        pools["first"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "last", "target": str(base["order"][-1])})
        pools["last"].append(item)

        for first_index, second_index in PAIR_INDICES:
            pools["pairwise"].append(make_pairwise_item(base, first_index, second_index))
            pools["pairwise"].append(make_pairwise_item(base, second_index, first_index))
    return pools


def sample_records(records, count, rng):
    indices = rng.integers(0, len(records), size=count)
    return [records[int(index)] for index in indices]


def sample_pairwise_records(records, count, rng):
    target_1 = [record for record in records if record["target"] == "1"]
    target_2 = [record for record in records if record["target"] == "2"]
    half = count // 2
    sampled = sample_records(target_1, half + count % 2, rng) + sample_records(target_2, half, rng)
    rng.shuffle(sampled)
    return sampled


def label_distribution(records, task_type="pairwise"):
    selected = [record for record in records if record["task_type"] == task_type]
    counts = pd.Series([record["target"] for record in selected]).value_counts().to_dict()
    total = max(1, len(selected))
    return {
        "total": len(selected),
        "target_1": int(counts.get("1", 0)),
        "target_2": int(counts.get("2", 0)),
        "target_1_rate": counts.get("1", 0) / total,
        "target_2_rate": counts.get("2", 0) / total,
    }


def build_balanced_records(dataframe):
    pools = build_record_pools(dataframe)
    pool_dist = label_distribution(pools["pairwise"])
    assert abs(pool_dist["target_1_rate"] - 0.5) < 1e-9, pool_dist
    assert abs(pool_dist["target_2_rate"] - 0.5) < 1e-9, pool_dist

    base_total = int(math.ceil(len(pools["order"]) / TASK_RATIOS["order"]))
    rng = np.random.default_rng(SEED)
    merged = []
    for task, ratio in TASK_RATIOS.items():
        count = max(1, int(round(base_total * ratio)))
        if task == "pairwise":
            records = sample_pairwise_records(pools[task], count, rng)
        else:
            records = sample_records(pools[task], count, rng)
        merged.extend(records)
        print(task, "pool:", len(pools[task]), "sampled:", len(records))
    rng.shuffle(merged)

    sampled_dist = label_distribution(merged)
    pair_target_1_rate = sampled_dist["target_1_rate"]
    pair_target_2_rate = sampled_dist["target_2_rate"]
    print("pairwise sampled label distribution:", sampled_dist)
    assert abs(pair_target_1_rate - 0.5) < 0.05, sampled_dist
    assert abs(pair_target_2_rate - 0.5) < 0.05, sampled_dist
    return merged, pools


def rows_by_ids(dataframe, ids):
    ids = [str(value) for value in ids]
    subset = dataframe[dataframe["Id"].isin(ids)].copy()
    order = {sample_id: index for index, sample_id in enumerate(ids)}
    subset["_split_order"] = subset["Id"].map(order)
    return subset.sort_values("_split_order").drop(columns=["_split_order"]).reset_index(drop=True)


def assert_unique_ids(name, ids):
    ids = [str(value) for value in ids]
    duplicated = pd.Series(ids).value_counts()
    duplicated = duplicated[duplicated > 1]
    assert duplicated.empty, f"{name} has duplicated Ids: {duplicated.head().to_dict()}"


def validate_split_ids(split_ids, all_ids):
    all_ids = set(str(value) for value in all_ids)
    for name, ids in split_ids.items():
        ids = [str(value) for value in ids]
        assert_unique_ids(name, ids)
        missing = sorted(set(ids) - all_ids)
        assert not missing, f"{name} has Ids not found in train.csv: {missing[:5]}"

    train_ids = set(split_ids["train_ids.json"])
    validation_ids = set(split_ids["validation_ids.json"])
    quick_ids = set(split_ids["quick50_ids.json"])
    tuning_ids = set(split_ids["tuning150_ids.json"])
    holdout_ids = set(split_ids["holdout150_ids.json"])

    assert train_ids.isdisjoint(validation_ids), "train_ids and validation_ids overlap"
    assert quick_ids <= validation_ids, "quick50_ids must be a subset of validation_ids"
    assert tuning_ids <= validation_ids, "tuning150_ids must be a subset of validation_ids"
    assert holdout_ids <= validation_ids, "holdout150_ids must be a subset of validation_ids"
    assert quick_ids.isdisjoint(tuning_ids), "quick50_ids and tuning150_ids overlap"
    assert quick_ids.isdisjoint(holdout_ids), "quick50_ids and holdout150_ids overlap"
    assert tuning_ids.isdisjoint(holdout_ids), "tuning150_ids and holdout150_ids overlap"


def derive_eval_split_ids(validation_ids):
    validation_shuffle = [str(value) for value in validation_ids]
    rng = np.random.default_rng(SEED)
    rng.shuffle(validation_shuffle)
    quick50_ids = validation_shuffle[:min(QUICK_EVAL_ROWS, len(validation_shuffle))]
    remaining = validation_shuffle[len(quick50_ids):]
    tuning150_ids = remaining[:min(TUNING_EVAL_ROWS, len(remaining))]
    holdout150_ids = remaining[len(tuning150_ids):len(tuning150_ids) + min(HOLDOUT_EVAL_ROWS, max(0, len(remaining) - len(tuning150_ids)))]
    assert len(quick50_ids) == min(QUICK_EVAL_ROWS, len(validation_shuffle)), "quick split could not be created"
    assert len(tuning150_ids) == min(TUNING_EVAL_ROWS, max(0, len(validation_shuffle) - len(quick50_ids))), "tuning split could not be created"
    assert len(holdout150_ids) == min(HOLDOUT_EVAL_ROWS, max(0, len(validation_shuffle) - len(quick50_ids) - len(tuning150_ids))), "holdout split could not be created"
    return quick50_ids, tuning150_ids, holdout150_ids


def make_or_load_split_ids(dataframe):
    required = ["train_ids.json", "validation_ids.json", "quick50_ids.json", "tuning150_ids.json", "holdout150_ids.json"]
    paths = {name: os.path.join(SPLIT_DIR, name) for name in required}
    split_ids = {}

    train_path = paths["train_ids.json"]
    validation_path = paths["validation_ids.json"]
    if os.path.exists(train_path) and os.path.exists(validation_path):
        split_ids["train_ids.json"] = [str(value) for value in load_json(train_path)]
        split_ids["validation_ids.json"] = [str(value) for value in load_json(validation_path)]
        print("Loaded fixed train/validation Id split from:", SPLIT_DIR)
    else:
        unique_ids = dataframe["Id"].unique().copy()
        rng = np.random.default_rng(SEED)
        rng.shuffle(unique_ids)
        valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
        split_ids["validation_ids.json"] = [str(value) for value in unique_ids[:valid_size]]
        split_ids["train_ids.json"] = [str(value) for value in unique_ids[valid_size:]]
        save_json(split_ids["train_ids.json"], train_path)
        save_json(split_ids["validation_ids.json"], validation_path)
        print("Created and saved fixed train/validation Id split to:", SPLIT_DIR)

    derived_quick, derived_tuning, derived_holdout = derive_eval_split_ids(split_ids["validation_ids.json"])
    derived = {
        "quick50_ids.json": derived_quick,
        "tuning150_ids.json": derived_tuning,
        "holdout150_ids.json": derived_holdout,
    }
    for name in ["quick50_ids.json", "tuning150_ids.json", "holdout150_ids.json"]:
        if os.path.exists(paths[name]):
            split_ids[name] = [str(value) for value in load_json(paths[name])]
        else:
            split_ids[name] = derived[name]
            save_json(split_ids[name], paths[name])
            print("Created missing eval split:", paths[name])

    validate_split_ids(split_ids, dataframe["Id"].astype(str).tolist())
    for name, ids in split_ids.items():
        save_json(ids, os.path.join(RUN_SPLIT_DIR, name))
    return split_ids


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

split_ids = make_or_load_split_ids(train_df)
training_df = rows_by_ids(train_df, split_ids["train_ids.json"])
validation_df = rows_by_ids(train_df, split_ids["validation_ids.json"])
quick50_df = rows_by_ids(train_df, split_ids["quick50_ids.json"])
tuning150_df = rows_by_ids(train_df, split_ids["tuning150_ids.json"])
holdout150_df = rows_by_ids(train_df, split_ids["holdout150_ids.json"])

if TRAIN_ROWS is not None:
    training_df = training_df.sample(n=min(TRAIN_ROWS, len(training_df)), random_state=SEED).reset_index(drop=True)
if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

train_records, train_pools = build_balanced_records(training_df)
valid_pools = build_record_pools(validation_df)

run_config = {
    "model_repo_id": MODEL_REPO_ID,
    "model_id": MODEL_ID,
    "qwen2_best_adapter_dir": QWEN2_BEST_ADAPTER_DIR,
    "qwen2_reference_run": "qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine",
    "output_dir": OUTPUT_DIR,
    "split_dir": SPLIT_DIR,
    "task_ratios": TASK_RATIOS,
    "task_loss_weights": TASK_LOSS_WEIGHTS,
    "learning_rate": LEARNING_RATE,
    "max_train_steps": MAX_TRAIN_STEPS,
    "save_steps": SAVE_STEPS,
    "seed": SEED,
    "train_rows": len(training_df),
    "validation_rows": len(validation_df),
    "quick50_rows": len(quick50_df),
    "tuning150_rows": len(tuning150_df),
    "holdout150_rows": len(holdout150_df),
    "pairwise_pool_label_distribution": label_distribution(train_pools["pairwise"]),
    "pairwise_sampled_label_distribution": label_distribution(train_records),
}
save_json(run_config, os.path.join(RUN_ROOT, "run_config.json"))
save_json(run_config, os.path.join(OUTPUT_DIR, "train_config.json"))

print("train/valid/quick/tuning/holdout:", len(training_df), len(validation_df), len(quick50_df), len(tuning150_df), len(holdout150_df))
print("expected qwen2 reference train/valid:", 8582, 953)
if TRAIN_ROWS is None and VALID_ROWS is None:
    assert len(training_df) == 8582, f"train split mismatch: {len(training_df)} != 8582"
    assert len(validation_df) == 953, f"validation split mismatch: {len(validation_df)} != 953"
print("train records:", len(train_records))


In [ ]:
# 4) Prompt builders, dataset, collator
def task_instruction(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image occurs first?\n"
            "If the first image occurs earlier, answer 1.\n"
            "If the second image occurs earlier, answer 2.\n"
            "Answer only 1 or 2."
        )
    if task_type == "first":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image represents the beginning of the story?\n"
            "Answer only the image number from 1 to 4."
        )
    if task_type == "last":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image represents the end of the story?\n"
            "Answer only the image number from 1 to 4."
        )
    if task_type == "order":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Compare the temporal relation between scenes and identify the likely first and last scenes.\n"
            "Using these cues, determine the complete chronological order.\n"
            "Output only the final ordered list, such as [1, 2, 3, 4]."
        )
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        content.append({"type": "text", "text": f"\nImage {idx}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


class LGTRefineDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


class LGTRefineCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids, target):
        ids = input_ids.tolist()
        labels = input_ids.clone()
        start = None
        prefix = self.assistant_prefix_ids
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            decoded_tail = self.tokenizer.decode(ids[-160:], skip_special_tokens=False)
            raise ValueError(
                "Assistant prefix was not found in the tokenized chat template. "
                "Do not train until the label mask is fixed. "
                f"target={target!r}, decoded_tail={decoded_tail!r}"
            )
        labels[:start] = -100
        labels[labels == self.tokenizer.pad_token_id] = -100
        return labels

    def __call__(self, batch):
        texts = []
        images = []
        task_types = []
        targets = []
        for example in batch:
            text = self.processor.apply_chat_template(make_messages(example, include_answer=True), tokenize=False, add_generation_prompt=False)
            texts.append(text)
            images.append([load_rgb(path) for path in example["image_paths"]])
            task_types.append(example["task_type"])
            targets.append(str(example["target"]))
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        labels = torch.stack([self._mask_prompt(row, target) for row, target in zip(encoded["input_ids"], targets)])
        valid_target_counts = labels.ne(-100).sum(dim=1)
        if (valid_target_counts == 0).any():
            raise ValueError(f"Assistant target tokens were not found: {valid_target_counts.tolist()}")
        decoded_targets = [
            self.tokenizer.decode(row[row.ne(-100)], skip_special_tokens=True).strip()
            for row in labels
        ]
        for decoded, target, task_type in zip(decoded_targets, targets, task_types):
            if str(target) not in decoded:
                raise ValueError(f"Masked label does not contain target. task={task_type}, target={target!r}, decoded={decoded!r}")
        encoded["labels"] = labels
        encoded["task_type"] = task_types
        return encoded


In [ ]:
# 5) Build Qwen3-VL-4B QLoRA trainer
class TaskLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        task_types = inputs.pop("task_type", None)
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view(shift_labels.size())
        mask = shift_labels.ne(-100)
        sample_losses = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        weights = torch.tensor(
            [TASK_LOSS_WEIGHTS.get(task, 1.0) for task in task_types],
            dtype=sample_losses.dtype,
            device=sample_losses.device,
        )
        loss = (sample_losses * weights).mean()
        logs = {}
        for task in sorted(set(task_types)):
            task_mask = torch.tensor([value == task for value in task_types], device=sample_losses.device)
            if task_mask.any():
                logs[f"train_{task}_loss"] = sample_losses[task_mask].mean().detach().float().item()
        if logs:
            self.log(logs)
        return (loss, outputs) if return_outputs else loss


def load_model_class():
    if Qwen3VLForConditionalGeneration is not None:
        return Qwen3VLForConditionalGeneration
    if AutoModelForImageTextToText is not None:
        return AutoModelForImageTextToText
    if AutoModelForVision2Seq is not None:
        return AutoModelForVision2Seq
    raise ImportError("No compatible Qwen3-VL model class found. Re-run the dependency cell with transformers>=4.57.0 and restart runtime.")


def discover_lora_targets(active_model):
    preferred = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    matched = set()
    module_lines = []
    for name, module in active_model.named_modules():
        module_lines.append(f"{name}\t{module.__class__.__module__}.{module.__class__.__name__}")
        leaf = name.rsplit(".", 1)[-1]
        if leaf in preferred:
            matched.add(leaf)
    with open(os.path.join(OUTPUT_DIR, "all_module_names.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(module_lines))
    selected = [name for name in ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] if name in matched]
    if not selected:
        raise RuntimeError("No LoRA target modules found. Inspect all_module_names.txt.")
    save_json(selected, os.path.join(OUTPUT_DIR, "lora_target_modules.json"))
    return selected


processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


# Fail fast before training if the chat template or label mask changed.
_mask_check_batch = [train_records[index] for index in range(min(8, len(train_records)))]
_mask_check = LGTRefineCollator(processor)(_mask_check_batch)
_mask_counts = _mask_check["labels"].ne(-100).sum(dim=1).tolist()
print("label mask token counts:", _mask_counts)
del _mask_check

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_cls = load_model_class()
base_model = model_cls.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

lora_target_modules = discover_lora_targets(base_model)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=lora_target_modules,
)
model = get_peft_model(base_model, lora_config)
model.config.use_cache = False

print("LoRA target modules:", lora_target_modules)
model.print_trainable_parameters()

training_argument_values = {
    "output_dir": OUTPUT_DIR,
    "max_steps": MAX_TRAIN_STEPS,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": 0.03,
    "max_grad_norm": 0.3,
    "fp16": False,
    "bf16": True,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "paged_adamw_8bit",
    "logging_steps": LOGGING_STEPS,
    "save_strategy": "steps",
    "save_steps": SAVE_STEPS,
    "save_total_limit": None,
    "report_to": "none",
    "remove_unused_columns": False,
    "dataloader_num_workers": 0,
    "seed": SEED,
    "data_seed": SEED,
    "num_train_epochs": 1.0,
}
training_args = TrainingArguments(**training_argument_values)

trainer = TaskLossTrainer(
    model=model,
    args=training_args,
    train_dataset=LGTRefineDataset(train_records),
    data_collator=LGTRefineCollator(processor),
)


In [ ]:
# 6) Train Qwen3-VL-4B LoRA from the base model.
trainer.train()
final_adapter_dir = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_dir)
processor.save_pretrained(final_adapter_dir)
processor.save_pretrained(OUTPUT_DIR)
print("saved:", OUTPUT_DIR)


In [ ]:
# 7) Evaluation helpers: direct order and pair/first/last probability decoding
def digit_token_id(digit):
    ids = processor.tokenizer.encode(str(digit), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"Digit {digit} tokenized to {ids}")
    return ids[0]


DIGIT_TOKEN_IDS = {digit: digit_token_id(digit) for digit in [1, 2, 3, 4]}


def checkpoint_name(path):
    return os.path.basename(os.path.normpath(path))


def find_checkpoint_dirs(include_initial=False):
    dirs = []
    for name in os.listdir(OUTPUT_DIR):
        path = os.path.join(OUTPUT_DIR, name)
        if name.startswith("checkpoint-") and os.path.exists(os.path.join(path, "adapter_config.json")):
            dirs.append(path)
    final_dir = os.path.join(OUTPUT_DIR, "final_adapter")
    if os.path.exists(os.path.join(final_dir, "adapter_config.json")):
        dirs.append(final_dir)

    def sort_key(path):
        match = re.findall(r"checkpoint-(\d+)", path)
        if match:
            return int(match[-1])
        return 10**9

    return sorted(dict.fromkeys(dirs), key=sort_key)


def model_device(active_model):
    return next(active_model.parameters()).device


def sanitize_generation_config(active_model):
    generation_config = active_model.generation_config
    generation_config.do_sample = False
    generation_config.temperature = None
    generation_config.top_p = None
    generation_config.top_k = None
    generation_config.num_beams = 1
    return active_model


def load_eval_model(adapter_dir):
    base = load_model_class().from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        local_files_only=MODEL_LOCAL_FILES_ONLY,
        trust_remote_code=True,
    )
    loaded = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
    sanitize_generation_config(loaded)
    loaded.eval()
    return loaded


def make_eval_example(row, task_type, pair=None):
    answer = [int(value) for value in row.get("Answer_list", [1, 2, 3, 4])]
    image_paths = row_image_paths(row, TRAIN_IMAGE_DIR)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order_to_sequence(answer),
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["first_index"] = a - 1
        example["second_index"] = b - 1
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    return example


@torch.no_grad()
def score_digit_candidates(active_model, example, candidates):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
    outputs = active_model(**inputs)
    last_pos = int(inputs["attention_mask"][0].sum().item()) - 1
    logits = outputs.logits[0, last_pos]
    token_ids = [DIGIT_TOKEN_IDS[int(candidate)] for candidate in candidates]
    probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
    processor.tokenizer.padding_side = old_padding_side
    return {int(candidate): float(prob) for candidate, prob in zip(candidates, probs)}


@torch.no_grad()
def generate_order(active_model, row, max_new_tokens=16):
    example = make_eval_example(row, "order")
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
    generated = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = generated[:, inputs["input_ids"].shape[1]:]
    output = processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    processor.tokenizer.padding_side = old_padding_side
    return parse_order_prediction(output), output


def order_ranks(order):
    return {int(image_number): position for position, image_number in enumerate(order)}


def pair_accuracy_from_orders(pred_order, gold_order):
    pred_ranks = order_ranks(pred_order)
    gold_ranks = order_ranks(gold_order)
    return np.mean([
        (pred_ranks[a] < pred_ranks[b]) == (gold_ranks[a] < gold_ranks[b])
        for a, b in itertools.combinations([1, 2, 3, 4], 2)
    ])


def order_metric_row(pred_order, gold_order):
    if pred_order is None:
        return {"exact_match": 0.0, "pair_accuracy": 0.0, "position_accuracy": 0.0, "valid_output": 0.0}
    return {
        "exact_match": float(pred_order == gold_order),
        "pair_accuracy": float(pair_accuracy_from_orders(pred_order, gold_order)),
        "position_accuracy": float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
        "valid_output": 1.0,
    }


In [ ]:
# 8) Probability cache and structured decoding
def extract_probability_cache(adapter_dir, rows, tag):
    ckpt = checkpoint_name(adapter_dir)
    cache_path = os.path.join(EVAL_DIR, f"{ckpt}_{tag}_task_probability_cache.json")
    if os.path.exists(cache_path):
        print("[SKIP]", cache_path)
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)

    eval_model = load_eval_model(adapter_dir)
    records = []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"{ckpt} {tag} probs"):
        answer = [int(value) for value in row["Answer_list"]]
        gold_order = order_to_sequence(answer)
        first_probs = score_digit_candidates(eval_model, make_eval_example(row, "first"), [1, 2, 3, 4])
        last_probs = score_digit_candidates(eval_model, make_eval_example(row, "last"), [1, 2, 3, 4])
        pair_probs = {}
        pair_correct = []
        for first_index, second_index in PAIR_INDICES:
            a, b = first_index + 1, second_index + 1
            probs = score_digit_candidates(eval_model, make_eval_example(row, "pairwise", pair=(a, b)), [1, 2])
            p_a_before_b = probs[1]
            pair_probs[f"{a}>{b}"] = float(p_a_before_b)
            pair_probs[f"{b}>{a}"] = float(1.0 - p_a_before_b)
            pred_first = a if p_a_before_b >= 0.5 else b
            gold_first = a if answer[first_index] < answer[second_index] else b
            pair_correct.append(int(pred_first == gold_first))

        direct_order, direct_text = generate_order(eval_model, row)
        records.append({
            "sample_id": str(row["Id"]),
            "gold_order": gold_order,
            "first_probs": {str(k): v for k, v in first_probs.items()},
            "last_probs": {str(k): v for k, v in last_probs.items()},
            "pair_probs": pair_probs,
            "direct_order": direct_order,
            "direct_text": direct_text,
            "pairwise_accuracy": float(np.mean(pair_correct)),
            "first_accuracy": float(max(first_probs, key=first_probs.get) == gold_order[0]),
            "last_accuracy": float(max(last_probs, key=last_probs.get) == gold_order[-1]),
        })

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    del eval_model
    gc.collect()
    torch.cuda.empty_cache()
    return records


def structured_score(sample, order, alpha, beta, gamma):
    eps = 1e-12
    pair_score = np.mean([
        math.log(float(sample["pair_probs"][f"{order[i]}>{order[j]}"]) + eps)
        for i in range(4)
        for j in range(i + 1, 4)
    ])
    first_score = math.log(float(sample["first_probs"][str(order[0])]) + eps)
    last_score = math.log(float(sample["last_probs"][str(order[-1])]) + eps)
    return alpha * pair_score + beta * first_score + gamma * last_score


def decode_structured(sample, alpha, beta, gamma):
    return list(max(PERMUTATIONS, key=lambda order: structured_score(sample, order, alpha, beta, gamma)))


def evaluate_decoding(samples, alpha=1.0, beta=1.0, gamma=1.0, mode="structured"):
    rows = []
    for sample in samples:
        gold = [int(value) for value in sample["gold_order"]]
        if mode == "direct":
            pred = sample["direct_order"]
        else:
            pred = decode_structured(sample, alpha, beta, gamma)
        metric = order_metric_row(pred, gold)
        metric.update({
            "sample_id": sample["sample_id"],
            "pred_order": pred,
            "gold_order": gold,
            "pairwise_accuracy": sample["pairwise_accuracy"],
            "first_accuracy": sample["first_accuracy"],
            "last_accuracy": sample["last_accuracy"],
            "first_last_both_correct": float(sample["first_accuracy"] == 1.0 and sample["last_accuracy"] == 1.0),
        })
        rows.append(metric)
    df = pd.DataFrame(rows)
    summary = {
        "exact_match": df["exact_match"].mean(),
        "pair_accuracy": df["pair_accuracy"].mean(),
        "position_accuracy": df["position_accuracy"].mean(),
        "valid_output_rate": df["valid_output"].mean(),
        "mean_pairwise_accuracy": df["pairwise_accuracy"].mean(),
        "mean_first_accuracy": df["first_accuracy"].mean(),
        "mean_last_accuracy": df["last_accuracy"].mean(),
        "first_last_both_correct_rate": df["first_last_both_correct"].mean(),
        "exact_given_first_last_correct": df.loc[df["first_last_both_correct"] == 1.0, "exact_match"].mean() if (df["first_last_both_correct"] == 1.0).any() else np.nan,
    }
    return summary, df


def grid_search(samples, checkpoint, tag):
    path = os.path.join(EVAL_DIR, f"{checkpoint}_{tag}_decoding_weight_search.csv")
    if os.path.exists(path):
        return pd.read_csv(path)
    rows = []
    for alpha, beta, gamma in itertools.product(ALPHAS, BETAS, GAMMAS):
        summary, _ = evaluate_decoding(samples, alpha=alpha, beta=beta, gamma=gamma, mode="structured")
        summary.update({"checkpoint": checkpoint, "tag": tag, "alpha": alpha, "beta": beta, "gamma": gamma, "decoding": "pair_first_last"})
        rows.append(summary)
    df = pd.DataFrame(rows).sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False).reset_index(drop=True)
    df.to_csv(path, index=False)
    return df


In [ ]:
# 9) Quick checkpoint evaluation
quick_rows = quick50_df.copy().reset_index(drop=True)
checkpoint_dirs = find_checkpoint_dirs(include_initial=False)
print("checkpoints:", [checkpoint_name(path) for path in checkpoint_dirs])

summary_path = os.path.join(EVAL_DIR, "all_checkpoint_metrics_quick.csv")
if os.path.exists(summary_path):
    summary_df = pd.read_csv(summary_path)
    completed = set(summary_df["checkpoint"].astype(str))
    summary_rows = summary_df.to_dict("records")
else:
    completed = set()
    summary_rows = []

for adapter_dir in checkpoint_dirs:
    ckpt = checkpoint_name(adapter_dir)
    if ckpt in completed:
        print("[SKIP]", ckpt)
        continue
    samples = extract_probability_cache(adapter_dir, quick_rows, tag=f"quick{len(quick_rows)}")
    direct_summary, direct_predictions = evaluate_decoding(samples, mode="direct")
    direct_predictions.to_csv(os.path.join(EVAL_DIR, f"{ckpt}_quick_direct_predictions.csv"), index=False)
    direct_summary.update({"checkpoint": ckpt, "tag": f"quick{len(quick_rows)}", "decoding": "direct", "alpha": np.nan, "beta": np.nan, "gamma": np.nan})
    summary_rows.append(direct_summary)

    search = grid_search(samples, ckpt, tag=f"quick{len(quick_rows)}")
    best_structured = search.iloc[0].to_dict()
    summary_rows.append(best_structured)
    pd.DataFrame(summary_rows).to_csv(summary_path, index=False)

summary_df = pd.DataFrame(summary_rows).sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False).reset_index(drop=True)
display(summary_df)
top_checkpoints = summary_df[summary_df["decoding"].eq("pair_first_last")].head(TOP_K_FULL_EVAL)["checkpoint"].tolist()
print("top structured checkpoints:", top_checkpoints)


In [ ]:
# 10) Tuning/holdout validation for top checkpoints
# Tuning selects structured decoding weights. Holdout is reported with those weights and is not used for grid search.
tuning_rows = tuning150_df.copy().reset_index(drop=True)
holdout_rows = holdout150_df.copy().reset_index(drop=True)
assert set(tuning_rows["Id"].astype(str)).isdisjoint(set(holdout_rows["Id"].astype(str))), "tuning and holdout overlap"

full_summary_path = os.path.join(EVAL_DIR, "all_checkpoint_metrics_full.csv")
if os.path.exists(full_summary_path):
    full_summary_df = pd.read_csv(full_summary_path)
    completed = set(full_summary_df["checkpoint"].astype(str) + "::" + full_summary_df["decoding"].astype(str))
    full_rows_out = full_summary_df.to_dict("records")
else:
    completed = set()
    full_rows_out = []

name_to_dir = {checkpoint_name(path): path for path in checkpoint_dirs}
for ckpt in top_checkpoints:
    adapter_dir = name_to_dir[ckpt]
    tuning_samples = extract_probability_cache(adapter_dir, tuning_rows, tag=f"tuning{len(tuning_rows)}")
    holdout_samples = extract_probability_cache(adapter_dir, holdout_rows, tag=f"holdout{len(holdout_rows)}")

    key = ckpt + "::direct"
    if key not in completed:
        direct_summary, direct_predictions = evaluate_decoding(holdout_samples, mode="direct")
        direct_predictions.to_csv(os.path.join(EVAL_DIR, f"{ckpt}_holdout_direct_predictions.csv"), index=False)
        direct_summary.update({
            "checkpoint": ckpt,
            "tag": f"holdout{len(holdout_rows)}",
            "decoding": "direct",
            "alpha": np.nan,
            "beta": np.nan,
            "gamma": np.nan,
            "selection_exact_match": np.nan,
            "selection_pair_accuracy": np.nan,
            "selection_position_accuracy": np.nan,
        })
        full_rows_out.append(direct_summary)
        pd.DataFrame(full_rows_out).to_csv(full_summary_path, index=False)

    key = ckpt + "::pair_first_last"
    if key not in completed:
        tuning_search = grid_search(tuning_samples, ckpt, tag=f"tuning{len(tuning_rows)}")
        best_weights = tuning_search.iloc[0].to_dict()
        holdout_summary, structured_predictions = evaluate_decoding(
            holdout_samples,
            alpha=best_weights["alpha"],
            beta=best_weights["beta"],
            gamma=best_weights["gamma"],
            mode="structured",
        )
        structured_predictions.to_csv(os.path.join(EVAL_DIR, f"{ckpt}_holdout_structured_predictions.csv"), index=False)
        holdout_summary.update({
            "checkpoint": ckpt,
            "tag": f"holdout{len(holdout_rows)}",
            "decoding": "pair_first_last",
            "alpha": float(best_weights["alpha"]),
            "beta": float(best_weights["beta"]),
            "gamma": float(best_weights["gamma"]),
            "selection_exact_match": float(best_weights["exact_match"]),
            "selection_pair_accuracy": float(best_weights["pair_accuracy"]),
            "selection_position_accuracy": float(best_weights["position_accuracy"]),
        })
        full_rows_out.append(holdout_summary)
        pd.DataFrame(full_rows_out).to_csv(full_summary_path, index=False)

full_summary_df = pd.DataFrame(full_rows_out)
structured_mask = full_summary_df["decoding"].eq("pair_first_last")
if structured_mask.any():
    ranked_df = full_summary_df.sort_values(
        ["selection_exact_match", "selection_pair_accuracy", "selection_position_accuracy", "exact_match", "pair_accuracy", "position_accuracy"],
        ascending=False,
    ).reset_index(drop=True)
else:
    ranked_df = full_summary_df.sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False).reset_index(drop=True)
display(ranked_df)

best = ranked_df.iloc[0].to_dict()
best_checkpoint = best["checkpoint"]
best_adapter_dir = name_to_dir[best_checkpoint]
print("BEST:", best)

os.makedirs(BEST_ADAPTER_DIR, exist_ok=True)
if not os.path.exists(os.path.join(BEST_ADAPTER_DIR, "adapter_config.json")):
    best_model = load_eval_model(best_adapter_dir)
    best_model.save_pretrained(BEST_ADAPTER_DIR)
    processor.save_pretrained(BEST_ADAPTER_DIR)
    del best_model
    gc.collect()
    torch.cuda.empty_cache()

best_config = {
    "checkpoint": best_checkpoint,
    "checkpoint_dir": best_adapter_dir,
    "decoding": best["decoding"],
    "alpha": None if pd.isna(best.get("alpha", np.nan)) else float(best["alpha"]),
    "beta": None if pd.isna(best.get("beta", np.nan)) else float(best["beta"]),
    "gamma": None if pd.isna(best.get("gamma", np.nan)) else float(best["gamma"]),
    "task_ratio": TASK_RATIOS,
    "task_loss_weights": TASK_LOSS_WEIGHTS,
    "model_repo_id": MODEL_REPO_ID,
    "split_dir": SPLIT_DIR,
    "selection_exact_match": None if pd.isna(best.get("selection_exact_match", np.nan)) else float(best["selection_exact_match"]),
    "selection_pair_accuracy": None if pd.isna(best.get("selection_pair_accuracy", np.nan)) else float(best["selection_pair_accuracy"]),
    "holdout_exact_match": float(best["exact_match"]),
    "holdout_pair_accuracy": float(best["pair_accuracy"]),
    "holdout_position_accuracy": float(best["position_accuracy"]),
}
with open(os.path.join(BEST_ADAPTER_DIR, "best_config.json"), "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)
shutil.copy2(full_summary_path, os.path.join(BEST_ADAPTER_DIR, "all_checkpoint_metrics.csv"))
print("best adapter saved:", BEST_ADAPTER_DIR)


In [ ]:
# 11) Test inference and submission
def sequence_to_answer(order):
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


def make_test_example(row, task_type, pair=None):
    image_paths = row_image_paths(row, TEST_IMAGE_DIR)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": [1, 2, 3, 4],
        "order": [1, 2, 3, 4],
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    return example


@torch.no_grad()
def generate_test_order(active_model, row, max_new_tokens=16):
    example = make_test_example(row, "order")
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
    generated = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = generated[:, inputs["input_ids"].shape[1]:]
    output = processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    processor.tokenizer.padding_side = old_padding_side
    return parse_order_prediction(output), output


@torch.no_grad()
def test_digit_probs(active_model, row, task_type, candidates, pair=None):
    example = make_test_example(row, task_type, pair=pair)
    return score_digit_candidates(active_model, example, candidates)


with open(os.path.join(BEST_ADAPTER_DIR, "best_config.json"), "r", encoding="utf-8") as f:
    best_config = json.load(f)

test_model = load_eval_model(BEST_ADAPTER_DIR)
submission_rows = []
test_cache = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="test inference"):
    if best_config["decoding"] == "direct":
        pred_order, text = generate_test_order(test_model, row)
        if pred_order is None:
            pred_order = [1, 2, 3, 4]
    else:
        first_probs = test_digit_probs(test_model, row, "first", [1, 2, 3, 4])
        last_probs = test_digit_probs(test_model, row, "last", [1, 2, 3, 4])
        pair_probs = {}
        for first_index, second_index in PAIR_INDICES:
            a, b = first_index + 1, second_index + 1
            probs = test_digit_probs(test_model, row, "pairwise", [1, 2], pair=(a, b))
            pair_probs[f"{a}>{b}"] = float(probs[1])
            pair_probs[f"{b}>{a}"] = float(1.0 - probs[1])
        sample = {
            "sample_id": str(row["Id"]),
            "first_probs": {str(k): v for k, v in first_probs.items()},
            "last_probs": {str(k): v for k, v in last_probs.items()},
            "pair_probs": pair_probs,
        }
        pred_order = decode_structured(sample, best_config["alpha"], best_config["beta"], best_config["gamma"])
        test_cache.append(sample | {"pred_order": pred_order})

    submission_rows.append({"Id": str(row["Id"]), "Answer": str(sequence_to_answer(pred_order))})

submission = pd.DataFrame(submission_rows)
submission.to_csv(SUBMIT_PATH, index=False)
with open(os.path.join(EVAL_DIR, "test_probability_cache.json"), "w", encoding="utf-8") as f:
    json.dump(test_cache, f, ensure_ascii=False, indent=2)
shutil.copy2(SUBMIT_PATH, os.path.join(BEST_ADAPTER_DIR, "submission.csv"))
display(submission.head())
print("submission saved:", SUBMIT_PATH)


## Output Paths

```text
/content/drive/MyDrive/SNU_AI_Challenge/qwen3vl_4b_4task_structured_v1/runs/{RUN_ID}/qwen3vl_4b_4task/
```

Main artifacts:

- `train_config.json`
- `../splits/train_ids.json`
- `../splits/validation_ids.json`
- `../splits/quick50_ids.json`
- `../splits/tuning150_ids.json`
- `../splits/holdout150_ids.json`
- `eval/all_checkpoint_metrics_quick.csv`
- `eval/all_checkpoint_metrics_full.csv`
- `eval/*_task_probability_cache.json`
- `eval/*_decoding_weight_search.csv`
- `best_adapter/best_config.json`
- `submission_qwen3vl_4b_4task.csv`

The shared fixed split is stored here:

```text
/content/drive/MyDrive/SNU_AI_Challenge/id_splits/qwen2vl_lgt_order_refine_20260714_003635/
```

Qwen2 reference best adapter:

```text
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/best_adapter
```
